# Praktikum Machine Learning: Classification
**Nama:** Zhazhe Lhanata Khusuma  
**Platform:** Jupyter Notebook (Anaconda / VS Code)  
**Sumber:** Kaggle

Notebook ini mencakup 9 dataset:
- **Binary:** Churn Modelling, Hotel Booking Demand, Chest X-Ray Images (Pneumonia).
- **Multiclass:** Flowers Recognition, Leaf Classification, Dog Breed Identification.
- **Multilabel:** MoA Prediction, Freesound Audio Tagging 2019, iMaterialist Fashion 2018.

Setiap eksperimen berdiri sendiri. Jika dataset belum ditemukan, sel akan memberi informasi dan melanjutkan tanpa menghentikan seluruh notebook. Dataset tidak disertakan di dalam file IPYNB; unduh dari Kaggle terlebih dahulu.

## 1. Tujuan dan alur
Tahapan eksperimen: loading data, data understanding, pemeriksaan missing values, preprocessing, EDA, train-test split, pemodelan, evaluasi, visualisasi, dan kesimpulan.

**Metrik:** accuracy, precision, recall, F1-score, confusion matrix untuk binary/multiclass. Untuk multilabel digunakan micro F1, macro F1, Hamming Loss, dan subset accuracy.

## 2. Instalasi library
Jalankan satu kali pada kernel/environment yang digunakan.

In [ ]:
%pip install -q numpy pandas matplotlib seaborn scikit-learn pillow scikit-image joblib kaggle

Library opsional:
- `tensorflow` untuk CNN Chest X-Ray.
- `librosa` dan `soundfile` untuk ekstraksi fitur audio Freesound.

In [ ]:
# Jalankan jika diperlukan:
# %pip install -q tensorflow
# %pip install -q librosa soundfile

## 3. Import library dan konfigurasi
Buat folder `datasets` di direktori notebook. Letakkan setiap dataset di subfolder sesuai nama yang ditentukan.

In [ ]:
from pathlib import Path
import os, json, warnings, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    f1_score, precision_score, recall_score, hamming_loss
)
from skimage.feature import hog

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

DATA_ROOT = Path("datasets")
DATA_ROOT.mkdir(exist_ok=True)
DATA_PATHS = {
    "churn": DATA_ROOT / "churn-modelling",
    "hotel": DATA_ROOT / "hotel-booking-demand",
    "chest_xray": DATA_ROOT / "chest-xray-pneumonia",
    "flowers": DATA_ROOT / "flowers-recognition",
    "leaf": DATA_ROOT / "leaf-classification",
    "dog": DATA_ROOT / "dog-breed-identification",
    "moa": DATA_ROOT / "lish-moa",
    "freesound": DATA_ROOT / "freesound-audio-tagging-2019",
    "imaterialist": DATA_ROOT / "imaterialist-fashion-2018",
}
for name, folder in DATA_PATHS.items():
    print(f"{name:14s}: {folder} | tersedia = {folder.exists()}")

## 4. Mengunduh dataset dari Kaggle

Buat API token melalui Kaggle → Account → Create New Token. Simpan `kaggle.json` pada lokasi konfigurasi Kaggle. Untuk kompetisi, pastikan sudah menekan tombol **Join Competition**.

Jalankan perintah berikut pada terminal Anaconda Prompt / VS Code (bukan pada sel Python biasa):

In [ ]:
# Perintah terminal:
# kaggle datasets download -d shrutimechlearn/churn-modelling -p datasets/churn-modelling --unzip
# kaggle datasets download -d jessemostipak/hotel-booking-demand -p datasets/hotel-booking-demand --unzip
# kaggle datasets download -d paultimothymooney/chest-xray-pneumonia -p datasets/chest-xray-pneumonia --unzip
# kaggle datasets download -d alxmamaev/flowers-recognition -p datasets/flowers-recognition --unzip
# kaggle competitions download -c leaf-classification -p datasets/leaf-classification
# kaggle competitions download -c dog-breed-identification -p datasets/dog-breed-identification
# kaggle competitions download -c lish-moa -p datasets/lish-moa
# kaggle competitions download -c freesound-audio-tagging-2019 -p datasets/freesound-audio-tagging-2019
# kaggle competitions download -c imaterialist-challenge-fashion-2018 -p datasets/imaterialist-fashion-2018
#
# Ekstrak file ZIP kompetisi ke folder masing-masing sebelum menjalankan eksperimen.

In [ ]:
# Periksa file yang sudah diunduh
for name, folder in DATA_PATHS.items():
    print("\\n---", name, "---")
    if folder.exists():
        for p in list(folder.rglob("*"))[:20]:
            if p.is_file():
                print(p.relative_to(folder))
    else:
        print("Folder belum ada.")

## 5. Fungsi bantu: EDA dan evaluasi

Fungsi ini dipakai bersama agar hasil evaluasi tiap eksperimen konsisten.

In [ ]:
def find_file(folder, candidates):
    folder = Path(folder)
    for name in candidates:
        found = list(folder.rglob(name)) if folder.exists() else []
        if found:
            return found[0]
    return None

def show_distribution(y, title):
    counts = pd.Series(y).value_counts()
    display(counts.to_frame("jumlah"))
    plt.figure(figsize=(9,4))
    counts.plot(kind="bar")
    plt.title(title); plt.xlabel("Kelas"); plt.ylabel("Jumlah")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout(); plt.show()

def evaluate_single(y_true, y_pred, title):
    print("Accuracy :", round(accuracy_score(y_true, y_pred), 4))
    print("Precision weighted:", round(precision_score(y_true, y_pred, average="weighted", zero_division=0), 4))
    print("Recall weighted   :", round(recall_score(y_true, y_pred, average="weighted", zero_division=0), 4))
    print("F1 weighted       :", round(f1_score(y_true, y_pred, average="weighted", zero_division=0), 4))
    print("\\nClassification report:")
    print(classification_report(y_true, y_pred, zero_division=0))
    labels = np.unique(np.concatenate([np.asarray(y_true), np.asarray(y_pred)]))
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    plt.figure(figsize=(max(6, len(labels)*0.55), max(4, len(labels)*0.45)))
    sns.heatmap(cm, annot=(len(labels) <= 15), fmt="d", cmap="Blues")
    plt.title(title); plt.xlabel("Prediksi"); plt.ylabel("Aktual")
    plt.tight_layout(); plt.show()

def run_tabular(df, target, title, drop_cols=None):
    if target not in df.columns:
        print("Target tidak ditemukan:", target, "| kolom:", df.columns.tolist())
        return None
    data = df.dropna(subset=[target]).copy()
    X = data.drop(columns=[target] + (drop_cols or []), errors="ignore")
    y = data[target].astype(str)
    X = X.loc[:, X.nunique(dropna=False) > 1]
    categorical = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
    numeric = X.select_dtypes(include=np.number).columns.tolist()
    transformers = []
    if numeric:
        transformers.append(("num", SimpleImputer(strategy="median"), numeric))
    if categorical:
        cat_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=2))
        ])
        transformers.append(("cat", cat_pipe, categorical))
    prep = ColumnTransformer(transformers)
    model = Pipeline([
        ("preprocess", prep),
        ("classifier", RandomForestClassifier(
            n_estimators=150, random_state=RANDOM_STATE,
            class_weight="balanced", n_jobs=-1, min_samples_leaf=2
        ))
    ])
    strat = y if y.value_counts().min() >= 2 else None
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=strat
    )
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    print("Jumlah data:", len(data), "| jumlah fitur:", X.shape[1])
    evaluate_single(y_test, pred, title)
    return model

def image_hog(path, size=(128,128)):
    img = Image.open(path).convert("RGB").resize(size)
    return hog(np.asarray(img), orientations=9, pixels_per_cell=(8,8),
               cells_per_block=(2,2), channel_axis=-1)

def load_class_folders(root, max_images=2000, size=(128,128)):
    X, y = [], []
    root = Path(root)
    if not root.exists():
        return np.asarray(X), np.asarray(y)
    for folder in sorted([p for p in root.iterdir() if p.is_dir()]):
        for path in folder.rglob("*"):
            if path.suffix.lower() in [".jpg", ".jpeg", ".png", ".bmp"]:
                try:
                    X.append(image_hog(path, size))
                    y.append(folder.name)
                except Exception:
                    pass
                if len(X) >= max_images:
                    return np.asarray(X), np.asarray(y)
    return np.asarray(X), np.asarray(y)

# A. Binary Classification

Binary classification memprediksi salah satu dari dua kelas. Contohnya pelanggan keluar/tidak keluar dan pemesanan hotel dibatalkan/tidak dibatalkan.

## A1. Churn Modelling
**Dataset:** [Kaggle Churn Modelling](https://www.kaggle.com/datasets/shrutimechlearn/churn-modelling)  
**Target:** `Exited` (0 = tidak keluar, 1 = keluar). Kolom identitas tidak dipakai sebagai fitur.

In [ ]:
file = find_file(DATA_PATHS["churn"], ["Churn_Modelling.csv", "churn_modelling.csv"])
if file:
    churn = pd.read_csv(file)
    display(churn.head())
    print("Ukuran:", churn.shape)
    display(churn.isna().sum().to_frame("missing"))
    show_distribution(churn["Exited"], "Distribusi Churn")
    churn_model = run_tabular(churn, "Exited", "Churn Modelling - Confusion Matrix",
                              ["RowNumber", "CustomerId", "Surname"])
    if churn_model:
        joblib.dump(churn_model, "model_churn.joblib")
else:
    print("Churn_Modelling.csv belum ditemukan di datasets/churn-modelling.")

## A2. Hotel Booking Demand
**Dataset:** [Kaggle Hotel Booking Demand](https://www.kaggle.com/datasets/jessemostipak/hotel-booking-demand)  
**Target:** `is_canceled` (0 = tidak dibatalkan, 1 = dibatalkan). `reservation_status` dan tanggal status dihapus untuk menghindari kebocoran target.

In [ ]:
file = find_file(DATA_PATHS["hotel"], ["hotel_bookings.csv"])
if file:
    hotel = pd.read_csv(file)
    display(hotel.head())
    print("Ukuran:", hotel.shape)
    display(hotel.isna().sum().sort_values(ascending=False).head(15).to_frame("missing"))
    show_distribution(hotel["is_canceled"], "Distribusi Pembatalan Hotel")
    hotel_model = run_tabular(hotel, "is_canceled", "Hotel Booking - Confusion Matrix",
                              ["reservation_status", "reservation_status_date"])
    if hotel_model:
        joblib.dump(hotel_model, "model_hotel_booking.joblib")
else:
    print("hotel_bookings.csv belum ditemukan.")

## A3. Chest X-Ray Images (Pneumonia)
**Dataset:** [Kaggle Chest X-Ray Images](https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia)  
Citra dikelompokkan dalam folder `NORMAL` dan `PNEUMONIA`. Eksperimen berikut memakai transfer learning MobileNetV2 jika TensorFlow terpasang.

**Peringatan:** ini hanya latihan machine learning, bukan alat diagnosis medis.

In [ ]:
xray_root = DATA_PATHS["chest_xray"]
if xray_root.exists():
    for split in ["train", "val", "test"]:
        d = xray_root / split
        if d.exists():
            print(split, {p.name: len(list(p.glob("*"))) for p in d.iterdir() if p.is_dir()})
else:
    print("Folder dataset Chest X-Ray belum ditemukan.")

In [ ]:
try:
    import tensorflow as tf
    from tensorflow.keras import layers, models
    train_dir, test_dir = xray_root / "train", xray_root / "test"
    if train_dir.exists() and test_dir.exists():
        size, batch = (160,160), 32
        train_ds = tf.keras.utils.image_dataset_from_directory(
            train_dir, image_size=size, batch_size=batch, label_mode="binary", seed=RANDOM_STATE)
        test_ds = tf.keras.utils.image_dataset_from_directory(
            test_dir, image_size=size, batch_size=batch, label_mode="binary", shuffle=False)
        preprocess = tf.keras.applications.mobilenet_v2.preprocess_input
        train_ds = train_ds.map(lambda x,y: (preprocess(x),y)).prefetch(tf.data.AUTOTUNE)
        test_ds = test_ds.map(lambda x,y: (preprocess(x),y)).prefetch(tf.data.AUTOTUNE)
        base = tf.keras.applications.MobileNetV2(
            input_shape=size+(3,), include_top=False, weights="imagenet")
        base.trainable = False
        inputs = layers.Input(shape=size+(3,))
        x = base(inputs, training=False)
        x = layers.GlobalAveragePooling2D()(x)
        x = layers.Dropout(0.25)(x)
        outputs = layers.Dense(1, activation="sigmoid")(x)
        xray_model = models.Model(inputs, outputs)
        xray_model.compile(optimizer="adam", loss="binary_crossentropy",
                           metrics=["accuracy", tf.keras.metrics.Precision(), tf.keras.metrics.Recall()])
        history = xray_model.fit(train_ds, validation_data=test_ds, epochs=5)
        print("Evaluasi test:", xray_model.evaluate(test_ds))
        xray_model.save("model_chest_xray.keras")
        plt.plot(history.history["accuracy"], label="train")
        plt.plot(history.history["val_accuracy"], label="validation")
        plt.title("Chest X-Ray Accuracy"); plt.xlabel("Epoch"); plt.ylabel("Accuracy")
        plt.legend(); plt.show()
    else:
        print("Folder train/test tidak ditemukan.")
except ImportError:
    print("TensorFlow belum terpasang. Untuk menjalankan CNN: %pip install tensorflow")
except Exception as e:
    print("CNN tidak dapat dijalankan:", repr(e))

# B. Multiclass Classification

Multiclass classification memilih satu kelas dari lebih dari dua kelas, misalnya jenis bunga, spesies daun, atau ras anjing.

## B1. Flowers Recognition
**Dataset:** [Kaggle Flowers Recognition](https://www.kaggle.com/datasets/alxmamaev/flowers-recognition)  
Folder dataset biasanya berisi subfolder kelas bunga. Citra diekstrak menjadi fitur HOG lalu diklasifikasikan menggunakan Linear SVM. Sampel dibatasi agar lebih ringan.

In [ ]:
flower_X, flower_y = np.asarray([]), np.asarray([])
if DATA_PATHS["flowers"].exists():
    # Cari folder yang berisi subfolder kelas bunga.
    candidates = [DATA_PATHS["flowers"]] + [p for p in DATA_PATHS["flowers"].rglob("*") if p.is_dir()]
    for root in candidates:
        subdirs = [p for p in root.iterdir() if p.is_dir()]
        if len(subdirs) >= 2:
            X_try, y_try = load_class_folders(root, max_images=2000)
            if len(y_try) > 0:
                flower_X, flower_y = X_try, y_try
                print("Folder kelas:", root)
                break
if len(flower_y) > 0:
    show_distribution(flower_y, "Distribusi Kelas Bunga")
    Xtr, Xte, ytr, yte = train_test_split(
        flower_X, flower_y, test_size=0.2, random_state=RANDOM_STATE, stratify=flower_y)
    flower_model = LinearSVC(class_weight="balanced", random_state=RANDOM_STATE)
    flower_model.fit(Xtr, ytr)
    evaluate_single(yte, flower_model.predict(Xte), "Flowers - Confusion Matrix")
    joblib.dump(flower_model, "model_flowers_hog_svm.joblib")
else:
    print("Folder kelas bunga belum ditemukan. Periksa struktur dataset.")

## B2. Leaf Classification
**Dataset:** [Kaggle Leaf Classification](https://www.kaggle.com/c/leaf-classification)  
**Target:** `species`. Fitur berupa pengukuran bentuk daun. Kolom `id` tidak digunakan.

In [ ]:
file = find_file(DATA_PATHS["leaf"], ["train.csv"])
if file:
    leaf = pd.read_csv(file)
    display(leaf.head())
    print("Ukuran:", leaf.shape)
    show_distribution(leaf["species"], "Distribusi Spesies Daun")
    leaf_model = run_tabular(leaf, "species", "Leaf Classification - Confusion Matrix", ["id"])
    if leaf_model:
        joblib.dump(leaf_model, "model_leaf.joblib")
else:
    print("train.csv Leaf belum ditemukan. Ekstrak ZIP kompetisi terlebih dahulu.")

## B3. Dog Breed Identification
**Dataset:** [Kaggle Dog Breed Identification](https://www.kaggle.com/c/dog-breed-identification)  
Label terdapat pada `labels.csv` (kolom `id` dan `breed`), gambar berada pada folder `train`. Baseline memakai HOG + Linear SVM dan maksimal 2.000 gambar.

In [ ]:
dog_root = DATA_PATHS["dog"]
labels_file = find_file(dog_root, ["labels.csv"])
train_dir = dog_root / "train"
if labels_file and train_dir.exists():
    labels_df = pd.read_csv(labels_file)
    lookup = dict(zip(labels_df["id"].astype(str), labels_df["breed"].astype(str)))
    Xdog, ydog = [], []
    for path in list(train_dir.glob("*.jpg"))[:2000]:
        if path.stem in lookup:
            try:
                Xdog.append(image_hog(path, (96,96)))
                ydog.append(lookup[path.stem])
            except Exception:
                pass
    Xdog, ydog = np.asarray(Xdog), np.asarray(ydog)
    print("Citra terbaca:", len(ydog), "| kelas:", len(np.unique(ydog)))
    if len(ydog) > 5:
        show_distribution(ydog, "Distribusi Ras Anjing pada Sampel")
        strat = ydog if pd.Series(ydog).value_counts().min() >= 2 else None
        Xtr, Xte, ytr, yte = train_test_split(
            Xdog, ydog, test_size=0.2, random_state=RANDOM_STATE, stratify=strat)
        dog_model = LinearSVC(class_weight="balanced", random_state=RANDOM_STATE)
        dog_model.fit(Xtr, ytr)
        evaluate_single(yte, dog_model.predict(Xte), "Dog Breed - Confusion Matrix")
        joblib.dump(dog_model, "model_dog_breed_hog_svm.joblib")
    else:
        print("Sampel gambar belum cukup.")
else:
    print("labels.csv atau folder train belum ditemukan.")

# C. Multilabel Classification

Multilabel classification memungkinkan satu sampel memiliki lebih dari satu label. Evaluasi menggunakan micro F1, macro F1, Hamming Loss, dan subset accuracy.

## C1. Mechanisms of Action (MoA) Prediction
**Dataset:** [Kaggle MoA Prediction](https://www.kaggle.com/competitions/lish-moa)  
File umum: `train_features.csv` dan `train_targets_scored.csv`. Kolom `sig_id` digunakan untuk menggabungkan tabel, lalu tidak dipakai sebagai fitur.

In [ ]:
features_file = find_file(DATA_PATHS["moa"], ["train_features.csv"])
targets_file = find_file(DATA_PATHS["moa"], ["train_targets_scored.csv"])
if features_file and targets_file:
    fx = pd.read_csv(features_file)
    ty = pd.read_csv(targets_file)
    merged = fx.merge(ty, on="sig_id", how="inner")
    target_cols = [c for c in ty.columns if c != "sig_id"]
    X = merged.drop(columns=["sig_id"] + target_cols, errors="ignore").select_dtypes(include=np.number)
    Y = merged[target_cols].fillna(0).astype(int)
    X = X.fillna(X.median())
    print("Fitur:", X.shape, "| target:", Y.shape)
    display(Y.sum().sort_values(ascending=False).head(15).to_frame("jumlah positif"))
    Xtr, Xte, ytr, yte = train_test_split(X, Y, test_size=0.2, random_state=RANDOM_STATE)
    moa_model = ExtraTreesClassifier(n_estimators=100, max_features="sqrt",
                                     n_jobs=-1, random_state=RANDOM_STATE)
    moa_model.fit(Xtr, ytr)
    pred = moa_model.predict(Xte)
    print("Micro F1:", round(f1_score(yte, pred, average="micro", zero_division=0), 4))
    print("Macro F1:", round(f1_score(yte, pred, average="macro", zero_division=0), 4))
    print("Hamming Loss:", round(hamming_loss(yte, pred), 4))
    print("Subset Accuracy:", round(accuracy_score(yte, pred), 4))
    joblib.dump(moa_model, "model_moa_extratrees.joblib")
else:
    print("train_features.csv dan/atau train_targets_scored.csv belum ditemukan.")

## C2. Freesound Audio Tagging 2019
**Dataset:** [Kaggle Freesound Audio Tagging 2019](https://www.kaggle.com/competitions/freesound-audio-tagging-2019)  
File umum: `train_curated.csv` berisi `fname` dan `labels`; audio berada di folder `train_curated`. Fitur audio diringkas menggunakan mean dan standar deviasi MFCC. Jumlah sampel dibatasi 1.000.

In [ ]:
try:
    import librosa
    fs_root = DATA_PATHS["freesound"]
    csv_file = find_file(fs_root, ["train_curated.csv"])
    audio_dir = find_file(fs_root, ["train_curated"])
    if csv_file and audio_dir and Path(audio_dir).is_dir():
        fs = pd.read_csv(csv_file).head(1000)
        label_lists = fs["labels"].fillna("").apply(
            lambda s: [v.strip() for v in str(s).split(",") if v.strip()])
        from sklearn.preprocessing import MultiLabelBinarizer
        mlb = MultiLabelBinarizer()
        Yall = mlb.fit_transform(label_lists)
        feats, valid = [], []
        for i, fname in enumerate(fs["fname"]):
            try:
                signal, sr = librosa.load(Path(audio_dir)/fname, sr=22050, duration=5)
                mfcc = librosa.feature.mfcc(y=signal, sr=sr, n_mfcc=20)
                feats.append(np.r_[mfcc.mean(axis=1), mfcc.std(axis=1)])
                valid.append(i)
            except Exception:
                continue
        if len(feats) > 10:
            Xa, Ya = np.asarray(feats), Yall[valid]
            Xtr, Xte, ytr, yte = train_test_split(Xa, Ya, test_size=0.2, random_state=RANDOM_STATE)
            fs_model = ExtraTreesClassifier(n_estimators=100, n_jobs=-1, random_state=RANDOM_STATE)
            fs_model.fit(Xtr, ytr)
            pred = fs_model.predict(Xte)
            print("Audio berhasil diproses:", len(Xa))
            print("Micro F1:", round(f1_score(yte, pred, average="micro", zero_division=0), 4))
            print("Macro F1:", round(f1_score(yte, pred, average="macro", zero_division=0), 4))
            print("Hamming Loss:", round(hamming_loss(yte, pred), 4))
            joblib.dump({"model": fs_model, "label_binarizer": mlb}, "model_freesound.joblib")
        else:
            print("Jumlah audio yang berhasil diproses belum cukup.")
    else:
        print("train_curated.csv atau folder train_curated belum ditemukan.")
except ImportError:
    print("Librosa belum terpasang. Jalankan: %pip install librosa soundfile")
except Exception as e:
    print("Eksperimen audio gagal:", repr(e))

## C3. iMaterialist Challenge (Fashion) at FGVC5
**Dataset:** [Kaggle iMaterialist Fashion 2018](https://www.kaggle.com/c/imaterialist-challenge-fashion-2018)  
File anotasi biasanya `train.json`, berisi `image_id` dan `label_id`; gambar ada di folder `train`. Setiap gambar dapat memiliki beberapa label. Baseline memakai HOG + ExtraTrees, maksimal 1.000 gambar.

In [ ]:
im_root = DATA_PATHS["imaterialist"]
json_file = find_file(im_root, ["train.json"])
img_dir = im_root / "train"
if json_file and img_dir.exists():
    with open(json_file, "r", encoding="utf-8") as f:
        im_data = json.load(f)
    ann = pd.DataFrame(im_data.get("annotations", []))
    print("Jumlah anotasi:", len(ann))
    if {"image_id", "label_id"}.issubset(ann.columns):
        grouped = ann.groupby("image_id")["label_id"].apply(list)
        ids = list(grouped.index)[:1000]
        label_ids = sorted({lab for image_id in ids for lab in grouped.loc[image_id]})
        label_index = {lab:i for i,lab in enumerate(label_ids)}
        Xim, Yim = [], []
        for image_id in ids:
            candidates = [img_dir/f"{image_id}.jpg", img_dir/f"{image_id}.jpeg", img_dir/f"{image_id}.png"]
            path = next((p for p in candidates if p.exists()), None)
            if path is None:
                continue
            try:
                Xim.append(image_hog(path, (96,96)))
                row = np.zeros(len(label_ids), dtype=np.uint8)
                for lab in grouped.loc[image_id]:
                    row[label_index[lab]] = 1
                Yim.append(row)
            except Exception:
                pass
        if len(Xim) > 10:
            Xim, Yim = np.asarray(Xim), np.asarray(Yim)
            Xtr, Xte, ytr, yte = train_test_split(Xim, Yim, test_size=0.2, random_state=RANDOM_STATE)
            im_model = ExtraTreesClassifier(n_estimators=100, n_jobs=-1, random_state=RANDOM_STATE)
            im_model.fit(Xtr, ytr)
            pred = im_model.predict(Xte)
            print("Gambar diproses:", len(Xim), "| jumlah label:", Yim.shape[1])
            print("Micro F1:", round(f1_score(yte, pred, average="micro", zero_division=0), 4))
            print("Macro F1:", round(f1_score(yte, pred, average="macro", zero_division=0), 4))
            print("Hamming Loss:", round(hamming_loss(yte, pred), 4))
            joblib.dump({"model": im_model, "label_ids": label_ids}, "model_imaterialist.joblib")
        else:
            print("Gambar yang cocok dengan anotasi terlalu sedikit.")
    else:
        print("Format train.json tidak memiliki image_id dan label_id.")
else:
    print("train.json atau folder train belum ditemukan.")

# 6. Ringkasan dan interpretasi hasil

| Dataset | Tipe | Target |
|---|---|---|
| Churn Modelling | Binary | `Exited` |
| Hotel Booking Demand | Binary | `is_canceled` |
| Chest X-Ray | Binary | `NORMAL` / `PNEUMONIA` |
| Flowers Recognition | Multiclass | Nama folder bunga |
| Leaf Classification | Multiclass | `species` |
| Dog Breed Identification | Multiclass | `breed` |
| MoA Prediction | Multilabel | Kolom target mekanisme aksi |
| Freesound Audio Tagging | Multilabel | `labels` |
| iMaterialist Fashion | Multilabel | `label_id` |

**Cara membaca hasil:**
- Accuracy tinggi belum tentu baik jika kelas tidak seimbang.
- Precision menunjukkan ketepatan prediksi suatu kelas.
- Recall menunjukkan kemampuan menemukan data dari kelas tersebut.
- F1-score menyeimbangkan precision dan recall.
- Micro F1 menghitung seluruh keputusan label secara agregat; macro F1 memberi bobot yang sama pada setiap label.
- Hamming Loss yang lebih kecil menunjukkan lebih sedikit kesalahan label.
- Confusion matrix membantu melihat kelas yang sering tertukar.

# 7. Kesimpulan

Binary classification digunakan ketika target hanya memiliki dua kelas. Multiclass classification digunakan ketika satu data masuk ke satu dari banyak kelas. Multilabel classification digunakan ketika satu data dapat mempunyai beberapa label sekaligus.

Setiap jenis dataset membutuhkan preprocessing berbeda. Data tabular diproses dengan imputasi dan One-Hot Encoding. Data gambar dapat diubah menjadi fitur HOG sebagai baseline atau dipelajari menggunakan CNN. Data audio dapat diubah menjadi fitur MFCC sebelum klasifikasi.

**Kesimpulan model terbaik harus ditulis setelah notebook dijalankan**, berdasarkan nilai metrik aktual. Jangan mengisi angka evaluasi dengan perkiraan. Hasil antar-dataset juga tidak dapat dibandingkan langsung karena karakteristik, jumlah kelas, ukuran, dan distribusi datanya berbeda.

## 8. Troubleshooting dan catatan laporan
1. File tidak ditemukan: periksa lokasi folder dan nama file setelah ekstraksi.
2. Kaggle API gagal: pastikan `kaggle.json` benar dan akses kompetisi telah disetujui.
3. RAM terbatas: kurangi batas jumlah gambar/audio pada sel terkait.
4. TensorFlow/Librosa tidak tersedia: instal library opsional atau lewati eksperimen tersebut.
5. Untuk laporan, catat jumlah data yang berhasil dibaca, pembagian train-test, parameter model, versi library, dan hasil evaluasi aktual.
6. Model HOG + SVM / ExtraTrees pada notebook ini adalah **baseline pembelajaran**, bukan jaminan skor kompetisi terbaik.